## SteadyCom

In [1]:
import reframed
from reframed import SteadyCom
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

import subprocess
import time

In [2]:
reframed.set_default_solver("cplex")

#### Load data and models

**Load relative abundance (E. coli)**

In [3]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [4]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [5]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/top_7_strains/"+path,
                                                        exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")

*Gut microbiome members*

In [6]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [7]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [8]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [9]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

### Run SteadyCom for E.coli with fixed abundance

In [10]:
relative_abundance_exp = pd.read_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",sep="\t",header=None)
relative_abundance_exp[2] = relative_abundance_exp[2]/relative_abundance_exp[2].sum()

### Run SteadyCom for E.coli 

### Run SteadyCom for Gut microbiome

In [11]:
constraints_gut ={}
for file in listdir("../data/Venturelli2018/generated/communities_steadiercom/"):                
    if not file.endswith(".tsv"):
        continue
    
    relative_abundance_exp_gut = pd.read_csv("../data/Venturelli2018/generated/communities_steadiercom/"+file,sep="\t",header=None)
    # make sure the relative abundance sums to 1
    extracted = pd.Series({"x_M_"+row[1]:row[2] for i,row in relative_abundance_exp_gut.iterrows()})
    extracted = extracted/extracted.sum()
    constraints_gut[relative_abundance_exp_gut.iloc[0,0]] =extracted.to_dict()

In [12]:
def SteadyComGutCommunities(fixed_relative_abundance=True):    
    cross_feeding_gut_comms = {}
    relative_abundance_gut_comms = {}
    growth_gut_comms = {}

    for community_id,relative_abundance_dict in constraints_gut.items():
        print(f"Started SteadyCom analysis for {community_id}")
        # Prepare for SteadyCom
        model_ids = [id_[4:] for id_ in relative_abundance_dict.keys()]
        models_specific = [model for id_, model in models_gut.items() if id_ in model_ids]
        
        comm_gut = reframed.Community(community_id=community_id,models=models_specific)
        env_gut.apply(comm_gut.merged_model,inplace=True,exclusive=True,warning=False)
        
        # Run SteadyCom
        time_1 = time.time()
        if fixed_relative_abundance:
            sol_steadycom_gut = SteadyCom(comm_gut,constraints=relative_abundance_dict)
        else:
            sol_steadycom_gut = SteadyCom(comm_gut)
        time_2 = time.time()
        print(f"\tRunning SteadyCom took: {(time_2 - time_1)/60:.2f} minutes")
        
        # Gather data
        cross_feeding_gut = pd.DataFrame(sol_steadycom_gut.cross_feeding())
        cross_feeding_gut.columns = ["donor","receiver","compound","rate?","mass_rate?"]
        relative_abundance_gut = sol_steadycom_gut.abundance
        growth_gut = sol_steadycom_gut.growth
        
        cross_feeding_gut_comms[community_id] = cross_feeding_gut
        relative_abundance_gut_comms[community_id] = relative_abundance_gut
        growth_gut_comms[community_id] = growth_gut
        
        time_3 = time.time()
        print(f"\tCollecting results took: {(time_3 - time_2)/60:.2f} minutes")
    return cross_feeding_gut_comms,relative_abundance_gut_comms,growth_gut_comms

#### Fixed abundance

#### Not fixed abundance

In [13]:
cross_feeding_gut_comms,relative_abundance_gut_comms,growth_gut_comms = SteadyComGutCommunities(fixed_relative_abundance=False)


Started SteadyCom analysis for ER-
	Running SteadyCom took: 4.25 minutes


/Users/idunmariaburgos/SteadierCom/reframed/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


Started SteadyCom analysis for DP-
	Running SteadyCom took: 0.43 minutes
Started SteadyCom analysis for PC-
	Running SteadyCom took: 8.55 minutes
Started SteadyCom analysis for EL-
	Running SteadyCom took: 4.15 minutes
Started SteadyCom analysis for BT-
	Running SteadyCom took: 6.86 minutes
Started SteadyCom analysis for BU-
	Running SteadyCom took: 5.27 minutes
Started SteadyCom analysis for NONE
	Running SteadyCom took: 6.57 minutes
Started SteadyCom analysis for BV-
	Running SteadyCom took: 3.27 minutes
Started SteadyCom analysis for CH-
	Running SteadyCom took: 2.49 minutes
Started SteadyCom analysis for BO-
	Running SteadyCom took: 0.38 minutes
Started SteadyCom analysis for BH-
	Running SteadyCom took: 4.71 minutes
Started SteadyCom analysis for FP-
	Running SteadyCom took: 3.20 minutes
Started SteadyCom analysis for CA-
	Running SteadyCom took: 3.04 minutes


In [14]:
pd.DataFrame({"growth":growth_gut_comms}).to_csv("../output/Venturelli2018/steadycom/growth.tsv",sep="\t")

pd.DataFrame(relative_abundance_gut_comms).to_csv("../output/Venturelli2018/steadycom/relative_abundance.tsv",sep="\t")

cross_feeding_gut_df = pd.concat(cross_feeding_gut_comms).reset_index().drop("level_1",axis=1)
cross_feeding_gut_df.columns= ['community', 'donor', 'receiver', 'compound', 'rate?','mass_rate?']
cross_feeding_gut_df.to_csv("../output/Venturelli2018/steadycom/cross_feeding.tsv",sep="\t",index=None)